# Fetching data

`transitio.fetch` turns an area of interest, or a place in the feed index,
into everything a routing tool needs — a cropped OpenStreetMap extract plus
validated GTFS feeds. This page goes through the call for an area and for a
place, the feeds that need an account, the trips several feeds repeat, and
handing the result to a routing tool.

## Fetch an area

Let's start by fetching the data for central Turku, given as a bounding box
of (west, south, east, north) coordinates in degrees:

In [1]:
import transitio

result = transitio.fetch((22.20, 60.42, 22.32, 60.48), when="2026-10-14")

Fetching 2 feeds for the area


[1/2] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


[2/2] f-vr~fi: cached copy reused


Comparing trips across 2 feeds


Fetching the OSM extract


Done in 14 s: 2 feeds delivered, 0 skipped


`fetch` shows its progress on stderr, a bar per download (a widget in
Jupyter with `pip install "transitio[notebook]"`, which adds ipywidgets);
`progress=False` turns it off. The warning says that most stops of the
delivered feeds lie outside the OpenStreetMap extract; the last section of
this page explains why.

The area of interest can be a place name (geocoded via Nominatim), a shapely
geometry, a GeoDataFrame/GeoSeries, or a `(minx, miny, maxx, maxy)` bounding
box in WGS84. Along the way, the call:

- downloads the smallest OpenStreetMap extract that contains the area, or
   a set of extracts smaller in total, merged into one (Geofabrik, BBBike or
   Movisda), and crops it to the area's bounding box (skipped with
   `osm=False`),
- takes the feeds the feed index lists for the places covering the area,
   or for the place — or, when no index is installed or its places cover
   less than half of the area's land, every GTFS feed whose bounding box
   meets the area's in the Mobility Database catalogue (official feeds
   first, then by spatial specificity), with a warning,
- downloads each feed, crops it to the area (its polygon, else its bounding
   box), repairs it when asked, and validates it with the canonical notice
   codes,
- leaves out of each delivered feed the trips an earlier delivered feed
   also runs,
- returns the artefact paths together with per-feed merged reports and a
   `(feed id, reason)` record for everything it skipped.

The result holds:

```python
result.osm_pbf     # pathlib.Path of the cropped .osm.pbf
result.feeds       # list of GTFS zip paths
result.paths       # {feed id: path}, in the order of result.feeds
result.reports     # per-feed merged validation reports (dicts)
result.repairs     # per-feed repair fix logs (empty without repair=True)
result.skipped     # [(feed id, reason), ...]
```

For Turku, two feeds were delivered and none skipped:

In [2]:
print(result.osm_pbf.name)
print(list(result.paths))
print(result.skipped)

bbox_22.2_60.42_22.32_60.48.osm.pbf
['f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja', 'f-vr~fi']
[]


Useful options:

```python
result = transitio.fetch(
    "Turku",
    when="2026-10-14",        # feeds must serve this day
    modes=["rail", "tram"],   # keep only feeds serving these modes
    repair=True,              # apply the gtfstidy-contract repair after the crop
    osm=False,                # skip the OSM extract; osm_pbf is then None
)
```

Without an API token for the Mobility Database, `fetch` takes each feed's
newest copy. When that copy does not run on `when`, a warning names the feed:
with a token, `fetch` could pick a dated copy from the Mobility Database.

## Fetch the feeds of an indexed place

`fetch(place=...)` selects the place's feeds by tier and crops them to
the place's boundary:

```python
augsburg = transitio.place("Augsburg")
result = transitio.fetch(
    place=augsburg,
    tiers=["local", "regional"],   # or exclude=["national", "international"]
    osm=False,
)
```

`feeds=["<feed id>", ...]`, or a recommendation from `place.recommend()`
(see [Choosing feeds](choosing_feeds.ipynb)), limits the call to those feeds
of the place, from any relevance category; an id the place does not have
raises `ValueError` before anything is downloaded. A recommendation also
gives the call its place and day. Let's fetch the two feeds recommended for
Tallinn:

In [3]:
tallinn = transitio.place("Tallinn")
rec = tallinn.recommend("2026-10-14")
tallinn_result = transitio.fetch(feeds=rec, osm=False)

Fetching 2 feeds for Tallinn


[1/2] f-ud-tallinn: cached copy reused


[2/2] f-mdb-3235: cached copy reused


Comparing trips across 2 feeds


Done in 3 s: 2 feeds delivered, 0 skipped


A feed that serves the place with only some of its routes is cropped to the
routes of the requested tiers. When that selection cannot be trusted — its
evidence was missing when the index was built, or the download no longer
matches it — `on_untrusted_selector` decides: `"auto"` (the default) skips
the feed when `exclude` was given and otherwise delivers it whole, `"whole"`
always delivers it whole, `"drop"` skips it and `"error"` raises
`StaleSelectorError`. National feeds, such as Germany's, are streamed
through the crop, so they fit in memory bounded by the area.

## Feeds that need an account

Some feeds need a free account with their provider. `fetch` sends their
credentials when they are set, with `transitio.credentials.set`, an
environment variable or `credentials={"<provider>": {"<field>": "..."}}`.
Without them, it reads the feed from the Mobility Database's hosted copy
when there is one, and otherwise skips the feed; either way the record
gives a reason such as `"protected feed: credentials missing for
trafiklab"`, followed by the feed's `access_instructions()`: who issues
the credentials and where to register. When a download with credentials
fails, the hosted copy is read without them.

## Trips that several feeds run

Feeds often publish the same trips: a city operator's buses also appear in
the regional and national feeds, under another agency name and with times a
minute apart. `fetch` delivers each such trip once. The feed earlier in the
selection record keeps it, for a place the order of `place.feeds()` (by
category, then relevance), and the later feed is delivered without it,
noted `"<n> repeated trips of <feed ids> left out"`; the `duplicate_trips`
column of `result.selection_table()` counts them. In Tallinn, the national
feed repeats many trips of the city's own feed:

In [4]:
table = tallinn_result.selection_table()
table[["feed_id", "decision", "duplicate_trips", "note"]]

,feed_id,decision,duplicate_trips,note
0,f-ud-tallinn,delivered,0,NaN
1,f-mdb-3235,delivered,8505,8505 repeated trips of f-ud-tallinn left out


Trips are compared as
`merge_feeds` compares them: the same route name and mode (bus, tram,
subway, rail or ferry, so a local bus coded 704 matches a bus coded 3), and
the same stops and times, or nearly (within 50 m and 3 minutes). With
`when`, the trips of that day are compared. Without it, a trip is left out
only when the earlier feeds run it on every date it runs, which reads every
feed's whole calendar: for the nine feeds of the Munich metro area the
comparison took about 60 seconds, against about 15 seconds for one day, and
memory peaked at about 6.6 GB against 2.7 GB. A feed is left out only when every
trip in scope, with `when` that day's, repeats a trip of an earlier feed; a
trip the comparison cannot read, such as one whose calendar cannot be read,
keeps its feed. The delivered feeds are separate feeds, so no transfer or
pathway links two of them. `duplicate_trips="exact"` leaves out exact
repeats only, and `duplicate_trips="keep"` compares nothing, so every feed
keeps all its trips.

## Hand the data to cafein or pyrosm

The result builds a routable [cafein](https://github.com/cafein-py/cafein)
network, or opens the extract in [pyrosm](https://pyrosm.readthedocs.io/).
Let's do both with the Turku result:

In [5]:
network = result.to_cafein()      # routable cafein.TransportNetwork
osm = result.to_pyrosm()          # pyrosm.OSM reader over the extract
walking = osm.get_network(network_type="walking")
print(network.stop_count, "stops;", len(walking), "walking network edges")

3230 stops; 23602 walking network edges


`to_cafein()` forwards keyword arguments to
`cafein.TransportNetwork.from_gtfs`, so e.g. `result.to_cafein(ultra=True)`
works as expected.

The crop keeps each trip that serves the area whole, with all its stops, so
some stops can lie beyond the area of the OSM extract (`result.osm_area`).
cafein may then find no walking network near such a stop and give it no
footpaths. A journey that starts and ends in the area is routed as before;
only a walking transfer at such a stop can be lost. The `stops_outside_osm` column of
`result.selection_table()` counts each delivered feed's located stops outside
`osm_area`, and `result.osm_note` sums them:

In [6]:
print(result.selection_table()[["feed_id", "stops_outside_osm"]])
print(result.osm_note)

                                             feed_id  stops_outside_osm
0  f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...               2210
1                                            f-vr~fi                307
OSM area: 2517 of 3230 located stops outside it


A located stop has usable
coordinates other than (0, 0), which stands for a missing position. The count
is geometric and can differ from the number cafein reports without
footpaths, either way: the extract spans the bounding box
of the area and cafein snaps a stop up to 1.6 km away, while a stop inside
the area can still lie far from any street or path.